In [2]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Reconnect to the existing DuckDB database (fast, no CSV reload)
con = duckdb.connect("../data/spotify.duckdb")

# Verify the data is still there
row_count = con.execute("SELECT COUNT(*) FROM charts").fetchone()[0]
print(f"Connected to DuckDB. Charts table has {row_count:,} rows.")

# Set a clean Plotly theme for all charts
import plotly.io as pio
pio.templates.default = "plotly_white"

Connected to DuckDB. Charts table has 26,173,514 rows.


In [3]:
# Load the Q1 result (market concentration by country)
q1 = pd.read_csv("../data/01_market_concentration.csv")

# Tag each country with a region for color-coding
region_map = {
    # Western Europe
    "Netherlands": "Western Europe", "Austria": "Western Europe",
    "Germany": "Western Europe", "Switzerland": "Western Europe",
    "Belgium": "Western Europe", "France": "Western Europe",
    "Italy": "Western Europe", "Portugal": "Western Europe",
    "Ireland": "Western Europe", "United Kingdom": "Western Europe",
    "Spain": "Western Europe",

    # Nordic
    "Sweden": "Nordic", "Norway": "Nordic", "Finland": "Nordic",
    "Denmark": "Nordic", "Iceland": "Nordic",

    # Eastern Europe
    "Poland": "Eastern Europe", "Czech Republic": "Eastern Europe",
    "Hungary": "Eastern Europe", "Slovakia": "Eastern Europe",
    "Lithuania": "Eastern Europe", "Latvia": "Eastern Europe",
    "Estonia": "Eastern Europe", "Greece": "Eastern Europe",
    "Turkey": "Eastern Europe",

    # Latin America
    "Brazil": "Latin America", "Mexico": "Latin America",
    "Argentina": "Latin America", "Colombia": "Latin America",
    "Chile": "Latin America", "Peru": "Latin America",
    "Ecuador": "Latin America", "Uruguay": "Latin America",
    "Paraguay": "Latin America", "Bolivia": "Latin America",
    "Costa Rica": "Latin America", "Panama": "Latin America",
    "Guatemala": "Latin America", "Honduras": "Latin America",
    "El Salvador": "Latin America", "Dominican Republic": "Latin America",

    # Asia
    "Japan": "Asia", "Singapore": "Asia", "Hong Kong": "Asia",
    "Taiwan": "Asia", "Indonesia": "Asia", "Malaysia": "Asia",
    "Philippines": "Asia", "Thailand": "Asia",

    # North America
    "United States": "North America", "Canada": "North America",

    # Oceania
    "Australia": "Oceania", "New Zealand": "Oceania",

    # Other
    "Global": "Global",
}

q1["region_group"] = q1["region"].map(region_map)

# Check that every country got tagged
unmapped = q1[q1["region_group"].isna()]
if len(unmapped) > 0:
    print("Countries without a region tag:")
    print(unmapped["region"].tolist())
else:
    print(f"All {len(q1)} countries tagged. Regions: {q1['region_group'].unique().tolist()}")

All 54 countries tagged. Regions: ['Asia', 'Nordic', 'Oceania', 'North America', 'Eastern Europe', 'Latin America', 'Western Europe', 'Global']


**Chart 1: Asia Listens to Fewer Artists, Europe Listens to More**

In [8]:
# Sort countries by region first, then by concentration within each region
# Order regions so the visual story reads top-to-bottom
region_order = ["Asia", "North America", "Oceania", "Eastern Europe",
                "Nordic", "Latin America", "Western Europe", "Global"]

q1["region_group"] = pd.Categorical(q1["region_group"], categories=region_order, ordered=True)
q1_sorted = q1.sort_values(["region_group", "pct_from_top10"], ascending=[True, False])

# Color palette per region
region_colors = {
    "Asia": "#d62728",            # red
    "North America": "#ff7f0e",   # orange
    "Oceania": "#bcbd22",         # olive
    "Eastern Europe": "#9467bd",  # purple
    "Nordic": "#17becf",          # cyan
    "Latin America": "#2ca02c",   # green
    "Western Europe": "#1f77b4",  # blue
    "Global": "#7f7f7f",          # gray
}

fig = px.bar(
    q1_sorted,
    x="pct_from_top10",
    y="region",
    color="region_group",
    color_discrete_map=region_colors,
    orientation="h",
    title="Market Concentration: % of Chart Slots Held by Top 10 Artists",
    labels={
        "pct_from_top10": "Share of chart slots from top 10 artists (%)",
        "region": "",
        "region_group": "Region"
    },
    height=1000,
    category_orders={"region": q1_sorted["region"].tolist()}
)

fig.update_layout(
    title=dict(font=dict(size=20, color="#222"), x=0.02, y=0.97),
    margin=dict(l=10, r=120, t=80, b=50),
    legend=dict(
        title="Region",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
        font=dict(size=11),
    ),
    yaxis=dict(tickfont=dict(size=11), automargin=True),
    xaxis=dict(tickfont=dict(size=11)),
    plot_bgcolor="white",
)

fig.update_traces(
    texttemplate="%{x:.1f}%",
    textposition="outside",
    textfont=dict(size=10, color="#444"),
)

fig.show()

# Save both formats
fig.write_html("../charts/01_market_concentration.html")
# fig.write_image("../charts/01_market_concentration.png", width=1200, height=1000, scale=2)  # disabled, kaleido + Python 3.14 incompatible
print("Saved chart to charts/01_market_concentration.html")

Saved chart to charts/01_market_concentration.html


**Chart 2: Song lifecycle has shrunk over time**

In [9]:
# Load the per-era summary CSV
q2_era = pd.read_csv("../data/02_song_lifecycle_by_era.csv")
print(q2_era)

                   era  num_hits  avg_days_to_climb  avg_days_at_top  \
0      Early (2017-18)        43               22.6             17.7   
1  Mid (2019-mid 2020)        42                5.7             12.7   
2   Late (mid 2020-21)        49                5.7             10.1   

   avg_total_days_on_chart  avg_days_after_peak  
0                    452.7                640.2  
1                    331.6                393.6  
2                    199.6                218.9  


In [10]:
# Reshape from wide to long format for plotting
q2_long = q2_era.melt(
    id_vars="era",
    value_vars=["avg_days_to_climb", "avg_days_at_top", "avg_total_days_on_chart"],
    var_name="metric",
    value_name="days"
)

# Friendlier metric labels
metric_labels = {
    "avg_days_to_climb": "Days to reach #1",
    "avg_days_at_top": "Days at #1",
    "avg_total_days_on_chart": "Total days on chart",
}
q2_long["metric"] = q2_long["metric"].map(metric_labels)

# Make sure eras display in chronological order
era_order = ["Early (2017-18)", "Mid (2019-mid 2020)", "Late (mid 2020-21)"]
q2_long["era"] = pd.Categorical(q2_long["era"], categories=era_order, ordered=True)
q2_long = q2_long.sort_values(["metric", "era"])

print(q2_long)

                   era               metric   days
3      Early (2017-18)           Days at #1   17.7
4  Mid (2019-mid 2020)           Days at #1   12.7
5   Late (mid 2020-21)           Days at #1   10.1
0      Early (2017-18)     Days to reach #1   22.6
1  Mid (2019-mid 2020)     Days to reach #1    5.7
2   Late (mid 2020-21)     Days to reach #1    5.7
6      Early (2017-18)  Total days on chart  452.7
7  Mid (2019-mid 2020)  Total days on chart  331.6
8   Late (mid 2020-21)  Total days on chart  199.6


In [11]:
# Color the eras from light (early) to dark (late) for an intuitive time gradient
era_colors = {
    "Early (2017-18)":      "#a8c5e8",  # light blue
    "Mid (2019-mid 2020)":  "#5c8ec9",  # medium blue
    "Late (mid 2020-21)":   "#1f4e79",  # dark blue
}

fig2 = px.bar(
    q2_long,
    x="metric",
    y="days",
    color="era",
    color_discrete_map=era_colors,
    barmode="group",
    title="Songs Reach #1 Faster and Burn Out Quicker",
    labels={
        "metric": "",
        "days": "Days (average)",
        "era": "Era"
    },
    height=550,
    text="days",
)

fig2.update_traces(
    texttemplate="%{text:.0f}",
    textposition="outside",
    textfont=dict(size=12, color="#333"),
)

fig2.update_layout(
    title=dict(font=dict(size=20, color="#222"), x=0.02, y=0.95),
    margin=dict(l=40, r=40, t=80, b=60),
    legend=dict(
        title="Era",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
    ),
    xaxis=dict(tickfont=dict(size=13)),
    yaxis=dict(tickfont=dict(size=11), gridcolor="#eee"),
    plot_bgcolor="white",
    bargap=0.25,
    bargroupgap=0.1,
)

# Annotation under the title
fig2.add_annotation(
    text="Among US chart-toppers, 2017 to 2021",
    showarrow=False,
    xref="paper", yref="paper",
    x=0.02, y=1.04,
    xanchor="left", yanchor="bottom",
    font=dict(size=12, color="#888"),
)

fig2.show()

# Save
fig2.write_html("../charts/02_song_lifecycle.html")
print("Saved chart to charts/02_song_lifecycle.html")

Saved chart to charts/02_song_lifecycle.html


**Chart 3: The world map of single-artist dominance**


In [12]:
# Load the Q3 result
q3 = pd.read_csv("../data/03_artist_dominance.csv")
print(q3.head())
print(f"\n{len(q3)} countries in Q3 data")

               region   top_artist  top_artist_appearances  country_total  \
0            Honduras    Bad Bunny                   16182       286269.0   
1  Dominican Republic    Bad Bunny                   17773       342318.0   
2         El Salvador    Bad Bunny                   13564       302106.0   
3         New Zealand        SIX60                   14405       358387.0   
4             Iceland  Hafdís Huld                    8324       212123.0   

   top_artist_share_pct  
0                  5.65  
1                  5.19  
2                  4.49  
3                  4.02  
4                  3.92  

54 countries in Q3 data


In [13]:
# Map our country names to ISO-3 codes for the choropleth
country_iso = {
    "United States": "USA", "Canada": "CAN", "Mexico": "MEX",
    "Brazil": "BRA", "Argentina": "ARG", "Colombia": "COL",
    "Chile": "CHL", "Peru": "PER", "Ecuador": "ECU",
    "Uruguay": "URY", "Paraguay": "PRY", "Bolivia": "BOL",
    "Costa Rica": "CRI", "Panama": "PAN", "Guatemala": "GTM",
    "Honduras": "HND", "El Salvador": "SLV", "Dominican Republic": "DOM",
    "United Kingdom": "GBR", "Ireland": "IRL", "France": "FRA",
    "Spain": "ESP", "Portugal": "PRT", "Italy": "ITA",
    "Germany": "DEU", "Austria": "AUT", "Switzerland": "CHE",
    "Belgium": "BEL", "Netherlands": "NLD",
    "Sweden": "SWE", "Norway": "NOR", "Denmark": "DNK",
    "Finland": "FIN", "Iceland": "ISL",
    "Poland": "POL", "Czech Republic": "CZE", "Hungary": "HUN",
    "Slovakia": "SVK", "Lithuania": "LTU", "Latvia": "LVA",
    "Estonia": "EST", "Greece": "GRC", "Turkey": "TUR",
    "Japan": "JPN", "Singapore": "SGP", "Hong Kong": "HKG",
    "Taiwan": "TWN", "Indonesia": "IDN", "Malaysia": "MYS",
    "Philippines": "PHL", "Thailand": "THA",
    "Australia": "AUS", "New Zealand": "NZL",
}

# Apply the mapping; "Global" gets dropped since it's not a real country
q3_map = q3[q3["region"] != "Global"].copy()
q3_map["iso_code"] = q3_map["region"].map(country_iso)

# Check that every country got a code
unmapped = q3_map[q3_map["iso_code"].isna()]
if len(unmapped) > 0:
    print("Countries missing an ISO code:")
    print(unmapped["region"].tolist())
else:
    print(f"All {len(q3_map)} countries mapped successfully")

All 53 countries mapped successfully


In [14]:
# Decide which artists get their own color
hero_artists = ["Bad Bunny", "Ed Sheeran", "Drake"]
q3_map["artist_group"] = q3_map["top_artist"].apply(
    lambda x: x if x in hero_artists else "Other regional artist"
)

# Display order so the legend reads nicely
artist_order = ["Bad Bunny", "Ed Sheeran", "Drake", "Other regional artist"]
q3_map["artist_group"] = pd.Categorical(q3_map["artist_group"], categories=artist_order, ordered=True)

# Print the count per group
print(q3_map["artist_group"].value_counts().sort_index())

artist_group
Bad Bunny                14
Ed Sheeran               15
Drake                     1
Other regional artist    23
Name: count, dtype: int64


In [15]:
# Color palette: hero artists get strong colors, "other" is muted
artist_colors = {
    "Bad Bunny":              "#e74c3c",   # red
    "Ed Sheeran":             "#3498db",   # blue
    "Drake":                  "#9b59b6",   # purple
    "Other regional artist":  "#bdc3c7",   # gray
}

fig3 = px.choropleth(
    q3_map,
    locations="iso_code",
    color="artist_group",
    hover_name="region",
    hover_data={
        "top_artist": True,
        "top_artist_share_pct": ":.2f",
        "iso_code": False,
        "artist_group": False,
    },
    color_discrete_map=artist_colors,
    title="Two Artists Own Most of the World",
    labels={"artist_group": "Top-charting artist"},
)

fig3.update_layout(
    title=dict(font=dict(size=22, color="#222"), x=0.02, y=0.95),
    margin=dict(l=10, r=10, t=80, b=10),
    legend=dict(
        title="Top-charting artist",
        orientation="v",
        yanchor="top",
        y=0.85,
        xanchor="left",
        x=0.01,
        font=dict(size=12),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#ccc",
        borderwidth=1,
    ),
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type="natural earth",
        bgcolor="white",
    ),
    height=600,
)

# Annotation under the title
fig3.add_annotation(
    text="Most-charting artist per country across the 2017 to 2021 Spotify top 200",
    showarrow=False,
    xref="paper", yref="paper",
    x=0.02, y=1.02,
    xanchor="left", yanchor="bottom",
    font=dict(size=12, color="#888"),
)

fig3.show()

# Save
fig3.write_html("../charts/03_artist_dominance.html")
print("Saved chart to charts/03_artist_dominance.html")

Saved chart to charts/03_artist_dominance.html
